# Safebooru 메타데이터 크롤링 (Selenium 우회 버전)
Safebooru 서버 차단을 피하기 위해 Chrome 웹 크롤링(Selenium)을 사용하여 데이터를 추출합니다.
- 이미지는 저장하지 않음 — 학습 시 배치 단위로 임시 다운로드
- 저장 컬럼: id, tags, file_url, sample_url, width, height
- 기존 데이터 파일 이어서 크롤링 가능

In [ ]:
import pandas as pd
import os
import time
import json
import re
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [ ]:
# --- 경로 설정 ---
data_dir = '../data'
metadata_path = os.path.join(data_dir, 'metadata.parquet')

# --- API 설정 ---
API_URL = 'https://safebooru.org/index.php'
LIMIT = 1000        # API 최대 반환 수
DELAY = 3.0         # 요청 간 딜레이 (초) - 브라우저 렌더링 대기를 위해 3초 권장
MAX_RETRIES = 3     # 페이지당 최대 재시도
SAVE_INTERVAL = 50  # N 페이지마다 중간 저장
MAX_PAGES = None    # 크롤링할 최대 페이지 수 (None이면 전체)

# --- 저장 컬럼 ---
KEEP_COLUMNS = ['id', 'tags', 'file_url', 'sample_url', 'width', 'height']

# --- Selenium 설정 ---
chrome_options = Options()
# 봇 탐지 우회를 위해 화면 창이 뜨는 기본 모드를 유지합니다.
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# 브라우저 실행
driver = webdriver.Chrome(options=chrome_options)

## 1. 전체 게시물 수 확인

In [ ]:
# 브라우저를 통해 카운트 정보가 담긴 XML 페이지에 접속
count_url = f"{API_URL}?page=dapi&s=post&q=index&limit=1"
driver.get(count_url)
time.sleep(1)

# 브라우저에 렌더링된 소스에서 정규식으로 전체 게시물 수 추출
match = re.search(r'count="(\d+)"', driver.page_source)
total_count = int(match.group(1)) if match else 0
total_pages = (total_count + LIMIT - 1) // LIMIT

if MAX_PAGES is not None:
    total_pages = min(total_pages, MAX_PAGES)

est_minutes = total_pages * DELAY / 60
print(f'전체 게시물 수: {total_count:,}')
print(f'크롤링 대상:    {total_pages:,} 페이지')
print(f'예상 소요 시간: 약 {est_minutes:.0f}분')

## 2. 메타데이터 크롤링

In [ ]:
start_pid = 0
existing_ids = set()
collected = 0

if os.path.exists(metadata_path):
    existing_df = pd.read_parquet(metadata_path)
    existing_ids = set(existing_df['id'].values)
    collected = len(existing_df)
    start_pid = max(0, collected // LIMIT - 2)
    print(f'기존 데이터 {collected:,}건. pid={start_pid}부터 이어서 크롤링합니다.')
else:
    existing_df = None
    print('처음부터 크롤링을 시작합니다.')

buffer = []
done = False
new_count = 0

try:
    tqdm._instances.clear()
except:
    pass

pbar = tqdm(range(start_pid, total_pages), initial=start_pid, total=total_pages, desc='크롤링')

for pid in pbar:
    for attempt in range(MAX_RETRIES):
        try:
            target_url = f"{API_URL}?page=dapi&s=post&q=index&limit={LIMIT}&pid={pid}&json=1"
            driver.get(target_url)
            
            # 페이지 로딩 및 차단 화면 우회를 위한 대기
            time.sleep(DELAY)
            
            # Chrome 브라우저는 JSON 결과값을 보통 pre 태그 안에 렌더링합니다.
            try:
                raw_text = driver.find_element(By.TAG_NAME, "pre").text
            except:
                raw_text = driver.find_element(By.TAG_NAME, "body").text
                
            if not raw_text.strip():
                raise Exception("Empty Response (서버 차단 의심)")

            posts = json.loads(raw_text)

            if not isinstance(posts, list) or not posts:
                done = True
                break

            for post in posts:
                if post['id'] not in existing_ids:
                    buffer.append({col: post.get(col) for col in KEEP_COLUMNS})
                    existing_ids.add(post['id'])
                    new_count += 1
            break

        except KeyboardInterrupt:
            tqdm.write('\n사용자에 의해 강제 중단되었습니다.')
            done = True
            break
            
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = (attempt + 1) * 30
                tqdm.write(f'pid={pid} 실패({e}), {wait_time}초 후 재시도... ({attempt+1}/{MAX_RETRIES})')
                time.sleep(wait_time)
            else:
                tqdm.write(f'pid={pid} 최종 실패: {e}')

    if done:
        break

    pbar.set_postfix({'수집': f'{collected + new_count:,}건', '신규': f'{new_count:,}건'})

    # 중간 저장
    if buffer and (pid + 1) % SAVE_INTERVAL == 0:
        new_df = pd.DataFrame(buffer)
        if existing_df is not None:
            new_df = pd.concat([existing_df, new_df], ignore_index=True)
        new_df = new_df.drop_duplicates(subset='id')
        new_df.to_parquet(metadata_path, index=False)
        existing_df = new_df
        buffer = []
        tqdm.write(f'중간 저장: {len(existing_df):,}건')

# 최종 저장 및 포맷팅 처리
if buffer:
    new_df = pd.DataFrame(buffer)
    if existing_df is not None:
        new_df = pd.concat([existing_df, new_df], ignore_index=True)
    new_df = new_df.drop_duplicates(subset='id')
    
    new_df['id'] = pd.to_numeric(new_df['id'], errors='coerce')
    new_df['width'] = pd.to_numeric(new_df['width'], errors='coerce')
    new_df['height'] = pd.to_numeric(new_df['height'], errors='coerce')
    new_df['tags'] = new_df['tags'].fillna('').astype(str)
    new_df['file_url'] = new_df['file_url'].fillna('').astype(str)
    new_df['sample_url'] = new_df['sample_url'].fillna('').astype(str)

    new_df.to_parquet(metadata_path, index=False)

# 최종 결과 요약 출력
df = pd.read_parquet(metadata_path)
file_size_mb = os.path.getsize(metadata_path) / (1024 * 1024)
print(f'\n크롤링 완료: {len(df):,}건 (신규 {new_count:,}건)')
print(f'파일 크기: {file_size_mb:.1f} MB')

# Selenium 브라우저 메모리 해제
driver.quit()

## 3. 확인

In [ ]:
df = pd.read_parquet(metadata_path)
print(f'총 {len(df):,}건')
print(f'컬럼: {list(df.columns)}')
print(f'URL 없는 행: {df["sample_url"].isna().sum():,}건')
print(f'태그 없는 행: {df["tags"].isna().sum():,}건')
df.head(3)